# 02 - Model Compression

`01` named precision as the single biggest lever for fitting a model inside a fixed
compute budget, and deferred the actual mechanics to this notebook. Here, we pull that
lever for real, on the same `yolov8n.pt` weights `perception_primer` already used - we
measure its size and inference latency as trained, quantize it to int8, and measure
both again. No fabricated numbers: every figure below is exactly what running this
notebook's cells produces, which is also why some of the results are less dramatic than
you might expect - that gap is the most important part of the lesson.

## Imports

In [ ]:
import os
import time

import numpy as np
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType


## Baseline: the Model as Trained

`YOLO("yolov8n.pt")` is the exact call `perception_primer/02-object-detection.ipynb`
made - if you've run that notebook already, this reuses the same cached weights file;
if not, `ultralytics` downloads it automatically the first time this cell runs.

In [ ]:
baseline = YOLO("yolov8n.pt")
baseline_size_mb = os.path.getsize(baseline.ckpt_path) / 1e6
print(f"yolov8n.pt size: {baseline_size_mb:.2f} MB")


## Measuring Latency, Honestly

Per `01`, edge inference runs one frame at a time - batch size 1 - so that's what we
benchmark here, not a batched throughput number that no robot's camera feed would ever
actually produce. We also warm up the model with a few throwaway predictions first
(the very first call always pays extra one-time setup cost that has nothing to do with
the model itself), then time a real run of predictions and report the mean and standard
deviation, not a single lucky number.

In [ ]:
TEST_IMAGES = [
    "../perception_primer/ref_imgs/yellow_ball.png",
    "../cv_primer/ref_imgs/balls_on_field.png",
]
N_RUNS = 30


def benchmark(weights_path, n_runs=N_RUNS, conf=0.15):
    """Runs single-image (batch size 1) inference n_runs times per test image.
    Returns (mean_ms, std_ms, detections_per_image).
    """
    model = YOLO(weights_path, task="detect")

    # Warmup - first call pays a one-time setup cost we don't want in the timing.
    for img in TEST_IMAGES:
        model.predict(img, conf=conf, verbose=False)

    times_s = []
    detections = {}
    for img in TEST_IMAGES:
        for _ in range(n_runs):
            start = time.perf_counter()
            result = model.predict(img, conf=conf, verbose=False)
            times_s.append(time.perf_counter() - start)
        boxes = result[0].boxes
        detections[img] = sorted(
            (round(c, 3) for c in boxes.conf.tolist()), reverse=True
        )

    times_ms = np.array(times_s) * 1000
    return times_ms.mean(), times_ms.std(), detections


baseline_mean_ms, baseline_std_ms, baseline_detections = benchmark(baseline.ckpt_path)
print(f"yolov8n.pt latency: {baseline_mean_ms:.2f} ms  (std {baseline_std_ms:.2f} ms)")
for img, confs in baseline_detections.items():
    print(f"  {img}: {confs}")


## Export: Leaving PyTorch Behind

`03` covers the export/convert/benchmark pipeline properly - for now, the one thing
worth knowing is that the quantization tooling we're about to use doesn't operate on a
PyTorch model directly. It operates on **ONNX** (Open Neural Network Exchange), an
interchange format most training frameworks can export to and most deployment
toolchains can read - the neutral middle format `03` is named after.

In [ ]:
onnx_path = baseline.export(format="onnx", imgsz=640, simplify=True)
onnx_size_mb = os.path.getsize(onnx_path) / 1e6
print(f"{onnx_path} size: {onnx_size_mb:.2f} MB")


## Quantizing to INT8

[ONNX Runtime's dynamic quantization](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html)
converts a model's weights to int8 with no calibration dataset required - it computes
activation ranges on the fly at inference time instead of ahead of time from sample
data. That "no calibration data" property is exactly why we're using it here: it's the
simplest real quantization technique to run, not necessarily the best one - the
**Try It Yourself** section below asks you to compare it against the calibrated
alternative ONNX Runtime's own documentation actually recommends for CNNs like this
one.

In [ ]:
int8_path = onnx_path.replace(".onnx", "_int8.onnx")
quantize_dynamic(onnx_path, int8_path, weight_type=QuantType.QUInt8)

int8_size_mb = os.path.getsize(int8_path) / 1e6
print(f"{int8_path} size: {int8_size_mb:.2f} MB")
print(f"Shrink vs. fp32 ONNX: {onnx_size_mb / int8_size_mb:.2f}x smaller")


## Re-Measuring: Size vs. Latency

Same `benchmark` function, same test images, same batch size 1 - the only thing that's
changed is which weights file we hand it.

In [ ]:
onnx_mean_ms, onnx_std_ms, onnx_detections = benchmark(onnx_path)
int8_mean_ms, int8_std_ms, int8_detections = benchmark(int8_path)

print(f"{'model':<20}{'size (MB)':>12}{'latency (ms)':>16}")
print(f"{'yolov8n.pt':<20}{baseline_size_mb:>12.2f}{baseline_mean_ms:>16.2f}")
print(f"{'yolov8n.onnx':<20}{onnx_size_mb:>12.2f}{onnx_mean_ms:>16.2f}")
print(f"{'yolov8n_int8.onnx':<20}{int8_size_mb:>12.2f}{int8_mean_ms:>16.2f}")

print()
for img in TEST_IMAGES:
    print(img)
    print(f"  fp32 ONNX detections: {onnx_detections[img]}")
    print(f"  int8 ONNX detections: {int8_detections[img]}")


## What Actually Happened

On the machine this notebook was written on, quantization shrank the model from 12.3 MB
(fp32 ONNX) to 3.5 MB (int8) - a reliable, repeatable ~3.5-4x reduction, every time you
run this notebook. Latency barely moved, and depending on your machine it may not move
at all, or even get slightly slower. **That's not a bug in this notebook - it's the
actual result, and it's the most important one here.**

Two things explain it, both already set up earlier in this primer:

- **Dynamic quantization isn't the right tool for a CNN's latency**, only for its size.
  ONNX Runtime's own documentation recommends dynamic quantization for RNNs and
  transformers, and *static* (calibrated) quantization for CNNs like YOLO - dynamic
  quantization still shrinks a CNN's weights on disk, but it doesn't restructure the
  convolution math enough to reliably speed it up. The **Try It Yourself** section below
  has you run the calibrated version and compare.
- **A generic laptop CPU doesn't have a specialized int8 execution path**, the way a
  real edge accelerator does. `01` named this exact gap: TOPS figures are usually quoted
  at int8 specifically because that's the precision edge silicon is built to accelerate.
  Benchmarking int8 quantization on a MacBook's CPU via a general-purpose runtime is
  benchmarking the wrong hardware for the question "does quantization make this faster
  on the edge" - which is precisely why `03` insists that benchmarking on your *actual
  target device* isn't optional busywork.

Look at the detection numbers too: on `balls_on_field.png`, every version of the model
agrees strongly on the single most confident detection (~0.78-0.83 confidence,
consistent through quantization) but disagrees more on the long tail of low-confidence
boxes. That's the accuracy cost of quantization in miniature - it doesn't uniformly
degrade every prediction, it erodes confidence fastest on the predictions that were
already borderline.

## Try It Yourself

1. Re-run the quantization cell with `weight_type=QuantType.QInt8` instead of
   `QuantType.QUInt8`. Does size or latency change?
2. Try Ultralytics' own calibrated INT8 export instead of the dynamic quantization used
   above: `baseline.export(format="onnx", int8=True, data="coco8.yaml")` (this
   downloads a small 8-image calibration set called `coco8` the first time you run it).
   Benchmark the result with the same `benchmark()` function. Does static, calibrated
   quantization close the latency gap that dynamic quantization didn't?
3. Change `TEST_IMAGES` to a single image repeated, and modify `benchmark()` to send
   all copies through `model.predict()` in **one call** instead of one at a time (a
   real batch, contradicting `01`'s "edge is batch size 1" framing on purpose). How
   much does per-image latency drop? This is the batching advantage `01` said cloud
   inference gets to use and edge usually can't.

## Resources

- [ONNX Runtime: Quantize ONNX Models](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html) -
  the dynamic vs. static quantization tradeoff referenced above, official documentation.
- [Ultralytics: Model Export](https://docs.ultralytics.com/modes/export) - the full
  `export()` API used above, including the calibrated `int8=True` path from
  **Try It Yourself**.
- [PyTorch: Quantization](https://docs.pytorch.org/docs/stable/quantization.html) - if
  you're quantizing a model you trained yourself in raw PyTorch rather than exporting
  through ONNX, this is the equivalent native API.